# TP Jour 3 — Spark SQL Avancé, Rigueur Statistique & Couche Gold

### Objectifs de la séance
1. Diagnostiquer l'**asymétrie (`SKEWNESS`)** de vos variables quantitatives et justifier l'emploi d'indicateurs robustes (**Médiane `PERCENTILE_APPROX`**, **IQR `P75 - P25`**, **P90**) plutôt que la seule moyenne arithmétique.
2. Écrire des requêtes **Spark SQL avancées** mobilisant des CTE (`WITH ...`) et des **fonctions de fenêtrage** (`DENSE_RANK`, `ROW_NUMBER`, cumul de **Pareto**, variation temporelle **`LAG`**).
3. Construire au moins **2 à 3 tables agrégées métier dans `/lakehouse/gold/`** prêtes à alimenter votre Dashboard Streamlit à faible latence.
4. Éviter le *Small Files Problem* sur HDFS et compacter votre table Silver avec `OPTIMIZE`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("J3_Analytics_SQL_Gold")
    .config("spark.driver.memory", "4g")  # 2 Go par défaut : insuffisant
    .getOrCreate()
)

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
print("Lignes disponibles dans v_silver :", spark.sql("SELECT COUNT(*) FROM v_silver").collect()[0][0])

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-34af631a-8230-4fef-b720-bcb89c993ec3;1.0
	confs: [default]


	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central


	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central


	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central
	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central


	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central
	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central


	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 999ms :: artifacts dl 34ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-pool;1.6 from central in [default]
	io.airlift#aircompressor;2.0.2 from central in [default]
	io.delta#delta-kernel-api;4.4.1 from central in [default]
	io.delta#delta-kernel-defaults;4.4.1 from central in [default]
	io.delta#delta-kernel-unitycatalog;4.4.1 from central in [default]
	io.delta#delta-spark_4.2_2.13;4.4.1 from cen

26/11/19 15:12:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/19 15:12:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Lignes disponibles dans v_silver : 5819079


## 1. Analyse d'Asymétrie : Moyenne vs Médiane & Écart Interquartile (IQR)

### Exercice 1.1 — Étudier la distribution de votre métrique quantitative principale
Écrivez une requête Spark SQL sur `v_silver` calculant sur votre variable continue principale :
- `COUNT(*)`, `AVG(...)`, `STDDEV(...)`, `SKEWNESS(...)`
- Les quantiles `P25`, `P50` (médiane), `P75`, `P90`, `P95` via `PERCENTILE_APPROX(col, q)` et `MAX(...)`.

Rédigez ensuite en Markdown **votre interprétation statistique** (écart moyenne/médiane, impact des valeurs extrêmes, choix de l'indicateur retenu pour votre dashboard).

In [2]:
df_dist_summary = spark.sql("""
    SELECT
        COUNT(*) AS nb_vols_avec_retard_renseigne,
        AVG(arrival_delay) AS moyenne_retard_min,
        STDDEV(arrival_delay) AS ecart_type_min,
        SKEWNESS(arrival_delay) AS asymetrie,
        PERCENTILE_APPROX(arrival_delay, 0.25) AS p25,
        PERCENTILE_APPROX(arrival_delay, 0.50) AS p50_mediane,
        PERCENTILE_APPROX(arrival_delay, 0.75) AS p75,
        PERCENTILE_APPROX(arrival_delay, 0.90) AS p90,
        PERCENTILE_APPROX(arrival_delay, 0.95) AS p95,
        MAX(arrival_delay) AS retard_max_min
    FROM v_silver
    WHERE arrival_delay IS NOT NULL
""")

df_dist_summary.show(truncate=False)

+-----------------------------+------------------+------------------+-----------------+---+-----------+---+---+---+--------------+
|nb_vols_avec_retard_renseigne|moyenne_retard_min|ecart_type_min    |asymetrie        |p25|p50_mediane|p75|p90|p95|retard_max_min|
+-----------------------------+------------------+------------------+-----------------+---+-----------+---+---+---+--------------+
|5714008                      |4.407057357987598 |39.271297093885984|6.502894504785266|-13|-5         |8  |34 |66 |1971          |
+-----------------------------+------------------+------------------+-----------------+---+-----------+---+---+---+--------------+



**Justification statistique**

Le coefficient d'asymétrie (skewness) vaut environ 6,5 : la distribution du retard à l'arrivée est très étirée vers la droite. La plupart des vols arrivent à l'heure ou en avance (médiane à −5 minutes), mais quelques retards très longs tirent la moyenne à +4,4 minutes. Le maximum atteint 1 971 minutes (près de 33 heures), ce qui gonfle aussi l'écart-type (39 minutes).

L'IQR est de 21 minutes (P75 − P25 = 8 − (−13)) : la moitié centrale des vols se situe entre 13 minutes d'avance et 8 minutes de retard. Pour représenter le retard d'un vol typique dans le dashboard, la médiane est donc plus pertinente que la moyenne. On la complète par le P90 (34 minutes) et le P95 (66 minutes) pour rendre visibles les gros retards, et par le taux de vols arrivés avec 15 minutes de retard ou plus (indicateur retenu en J2).

---
## 2. Table Gold #1 : Classement Métier & Loi de Pareto (`OVER ORDER BY ...`)

### Exercice 2.1 — Agrégation par segment/catégorie + Courbe cumulée de Pareto
Construisez en Spark SQL une première table Gold agrégée par une dimension métier pertinente de votre dataset incluant :
- Le volume d'observations, le total et les indicateurs robustes (**médiane**, **IQR**).
- Le rang via `DENSE_RANK() OVER (ORDER BY ... DESC)`.
- Le pourcentage cumulé du total (**analyse de Pareto 80/20**) via :
  `SUM(metric_total) OVER (ORDER BY metric_total DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) / SUM(metric_total) OVER ()`.

Sauvegardez-la avec `.coalesce(1)` au format **Delta** dans `/lakehouse/gold/<nom_gold_1>`.

In [3]:
# Minutes de retard (Pareto) et taux de retard par compagnie

GOLD_PARETO_PATH = "/lakehouse/gold/retards_par_compagnie"

df_gold_1 = spark.sql("""
    WITH stats_compagnies AS (
        SELECT
            airline,
            FIRST(airline_name) AS airline_name,
            COUNT(*) AS nb_vols,
            SUM(GREATEST(arrival_delay, 0)) AS metric_total,
            SUM(CAST(is_delayed_15 AS INT)) AS nb_retards_15,
            ROUND(100.0 * AVG(CAST(is_delayed_15 AS INT)), 2) AS taux_retard_15_pct,
            PERCENTILE_APPROX(arrival_delay, 0.50) AS retard_mediane_min,
            PERCENTILE_APPROX(arrival_delay, 0.25) AS p25_retard_min,
            PERCENTILE_APPROX(arrival_delay, 0.75) AS p75_retard_min,
            PERCENTILE_APPROX(arrival_delay, 0.90) AS p90_retard_min
        FROM v_silver
        WHERE airline IS NOT NULL
          AND arrival_delay IS NOT NULL
        GROUP BY airline
    )
    SELECT
        airline,
        airline_name,
        nb_vols,
        metric_total AS total_retard_positif_min,
        nb_retards_15,
        taux_retard_15_pct,
        retard_mediane_min,
        p75_retard_min - p25_retard_min AS iqr_retard_min,
        p90_retard_min,
        DENSE_RANK() OVER (ORDER BY metric_total DESC) AS rang_retard_total,
        DENSE_RANK() OVER (ORDER BY taux_retard_15_pct DESC) AS rang_taux_retard,
        ROUND(
            100.0 * SUM(metric_total) OVER (
                ORDER BY metric_total DESC, airline
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) / SUM(metric_total) OVER (),
            2
        ) AS pct_cumule_retards,
        ROUND(100.0 * nb_vols / SUM(nb_vols) OVER (), 2) AS part_des_vols_pct
    FROM stats_compagnies
    ORDER BY rang_retard_total
""")

df_gold_1.coalesce(1) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_PARETO_PATH)

df_gold_1.show(15, truncate=False)

26/11/19 14:44:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:44:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:44:56 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

+-------+----------------------------+-------+------------------------+-------------+------------------+------------------+--------------+--------------+-----------------+----------------+------------------+-----------------+
|airline|airline_name                |nb_vols|total_retard_positif_min|nb_retards_15|taux_retard_15_pct|retard_mediane_min|iqr_retard_min|p90_retard_min|rang_retard_total|rang_taux_retard|pct_cumule_retards|part_des_vols_pct|
+-------+----------------------------+-------+------------------------+-------------+------------------+------------------+--------------+--------------+-----------------+----------------+------------------+-----------------+
|WN     |Southwest Airlines Co.      |1242403|13849257                |236626       |19.05             |-4                |20            |33            |1                |8               |20.04             |21.74            |
|AA     |American Airlines Inc.      |712935 |8611910                 |130279       |18.27      

26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:06 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

**Interprétation**

- **Le Pareto suit surtout la taille des compagnies.** Les 6 premières (Southwest, American, Delta, Atlantic Southeast, United, Skywest) cumulent 76 % des minutes de retard, mais elles assurent aussi 78 % des vols. Il n'y a pas d'effet 80/20 marqué : aucune petite compagnie ne concentre une part démesurée du retard total.
- **Le taux de retard donne un autre classement.** Spirit (30 %) et Frontier (26 %), deux low-cost avec peu de vols, sont les moins fiables pour un voyageur, alors qu'elles ne pèsent que 2 % et 1,6 % des vols. Southwest, 1re en minutes de retard, n'est que 8e au taux (19 %). Delta (13,6 %), Alaska (13,0 %) et Hawaiian (11,3 %) sont les plus ponctuelles.
- **Médiane et IQR confirment l'asymétrie dans chaque compagnie** : toutes ont une médiane négative ou nulle (la moitié des vols arrive en avance), mais le P90 va de 16 minutes (Hawaiian) à 63 minutes (Spirit).
- Pour le dashboard, on affiche donc le taux de retard (qui compare des compagnies de tailles différentes) et on garde le Pareto pour montrer qui pèse dans le retard total du réseau.

## 3. Table Gold #2 : Dynamique Temporelle & Croissance avec `LAG()`

### Exercice 3.1 — Série temporelle et taux d'évolution période-sur-période
Agrégez votre table `v_silver` selon son axe temporel (jour, semaine, mois ou année selon votre sujet) et utilisez la fonction de fenêtrage **`LAG(metric, 1) OVER (ORDER BY periode)`** (ou une moyenne glissante `AVG(...) OVER (ORDER BY periode ROWS BETWEEN ... PRECEDING AND CURRENT ROW)`) pour mesurer la dynamique d'évolution.

Sauvegardez le résultat dans `/lakehouse/gold/<nom_gold_2>`.

In [4]:
# Variation NULL sans mois précédent

GOLD_TIME_PATH = "/lakehouse/gold/retards_mensuels"

df_gold_2 = spark.sql("""
    WITH mensuel AS (
        SELECT
            DATE_TRUNC('MONTH', flight_date) AS periode,
            MONTH(flight_date) AS mois,
            COUNT(*) AS nb_vols,
            COUNT(arrival_delay) AS nb_vols_avec_retard_renseigne,
            SUM(
                CASE
                    WHEN arrival_delay > 0 THEN arrival_delay
                    ELSE 0
                END
            ) AS total_retard_positif_min,
            ROUND(100.0 * AVG(CASE WHEN arrival_delay IS NOT NULL THEN CAST(is_delayed_15 AS INT) END), 2)
                AS taux_retard_15_pct,
            PERCENTILE_APPROX(arrival_delay, 0.50) AS retard_mediane_min,
            ROUND(100.0 * AVG(CAST(is_cancelled AS INT)), 2) AS taux_annulation_pct
        FROM v_silver
        GROUP BY DATE_TRUNC('MONTH', flight_date), MONTH(flight_date)
    ),
    avec_mois_precedent AS (
        SELECT
            *,
            LAG(total_retard_positif_min, 1) OVER (ORDER BY periode) AS retard_mois_precedent_min,
            LAG(taux_retard_15_pct, 1) OVER (ORDER BY periode) AS taux_retard_mois_precedent_pct,
            ROUND(AVG(taux_retard_15_pct) OVER (
                ORDER BY periode ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
            ), 2) AS taux_retard_moyenne_3_mois_pct
        FROM mensuel
    )
    SELECT
        periode,
        mois,
        nb_vols,
        nb_vols_avec_retard_renseigne,
        total_retard_positif_min,
        retard_mois_precedent_min,
        total_retard_positif_min - retard_mois_precedent_min
            AS variation_retard_min,
        CASE
            WHEN retard_mois_precedent_min > 0 THEN ROUND(
                100.0 * (total_retard_positif_min - retard_mois_precedent_min)
                / retard_mois_precedent_min,
                2
            )
            ELSE NULL
        END AS variation_retard_pct,
        taux_retard_15_pct,
        ROUND(taux_retard_15_pct - taux_retard_mois_precedent_pct, 2) AS variation_taux_retard_pts,
        taux_retard_moyenne_3_mois_pct,
        retard_mediane_min,
        taux_annulation_pct
    FROM avec_mois_precedent
    ORDER BY periode
""")

df_gold_2.coalesce(1) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_TIME_PATH)

df_gold_2.show(20, truncate=False)

26/11/19 14:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:11 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/19 14:45:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/19 14:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:17 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/11/19 14:45:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:45:20 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------------------+----+-------+-----------------------------+------------------------+-------------------------+--------------------+--------------------+------------------+-------------------------+------------------------------+------------------+-------------------+
|periode            |mois|nb_vols|nb_vols_avec_retard_renseigne|total_retard_positif_min|retard_mois_precedent_min|variation_retard_min|variation_retard_pct|taux_retard_15_pct|variation_taux_retard_pts|taux_retard_moyenne_3_mois_pct|retard_mediane_min|taux_annulation_pct|
+-------------------+----+-------+-----------------------------+------------------------+-------------------------+--------------------+--------------------+------------------+-------------------------+------------------------------+------------------+-------------------+
|2015-01-01 00:00:00|1   |469968 |457013                       |5997907                 |NULL                     |NULL                |NULL                |21.0              |NULL 

**Interprétation**

- **Deux pics de retard : février (23,4 %) et juin (23,5 %).** Février correspond aux tempêtes d'hiver (c'est aussi le mois record d'annulations, 4,8 %), juin aux orages d'été et au début des vacances. Le total de minutes de retard bondit de 33 % entre mai et juin.
- **Le creux est en septembre et octobre** (13,0 % et 12,4 %), entre la fin de l'été et les fêtes. Le retard remonte ensuite en décembre (20,6 %).
- **La moyenne glissante sur 3 mois lisse ces à-coups** : elle reste autour de 20 à 22 % de janvier à août, puis descend sous 15 % en octobre et novembre. La saison compte donc, mais moins que l'heure de départ ou la compagnie (conclusion de J2).
- Le total de minutes varie aussi avec le nombre de vols du mois : c'est pour cela qu'on suit en parallèle le taux de retard, comparable d'un mois à l'autre.

## 4. Table Gold #3 (Axe complémentaire libre) & Compaction `OPTIMIZE`

### Exercice 4.1 — Produire un 3e axe d'analyse (`ROW_NUMBER() OVER (PARTITION BY ...)`) et compacter Silver
1. Créez une 3e table Gold répondant à une question spécifique de votre problématique (ex. top sous-catégorie par région/groupe via `ROW_NUMBER() OVER (PARTITION BY groupe ORDER BY ... DESC) = 1`).
2. Exécutez `OPTIMIZE delta.'{SILVER_MAIN_PATH}'` et expliquez pourquoi la multiplication de micro-fichiers pénalise le NameNode HDFS.

In [5]:
# Compagnie la plus en retard par aéroport (au taux, pas au total de minutes)

GOLD_TOP_AIRLINE_PATH = "/lakehouse/gold/top_compagnie_retards_par_aeroport"

df_gold_3 = spark.sql("""
    WITH vols AS (
        SELECT *
        FROM v_silver
        WHERE airport_code_valid
          AND origin_airport IS NOT NULL
          AND airline IS NOT NULL
          AND arrival_delay IS NOT NULL
    ),
    aeroports AS (
        SELECT
            origin_airport,
            FIRST(origin_city) AS origin_city,
            FIRST(origin_state) AS origin_state,
            FIRST(origin_lat) AS origin_lat,
            FIRST(origin_lon) AS origin_lon,
            COUNT(*) AS nb_vols_aeroport,
            ROUND(100.0 * AVG(CAST(is_delayed_15 AS INT)), 2) AS taux_retard_aeroport_pct
        FROM vols
        GROUP BY origin_airport
        HAVING COUNT(*) >= 1000
    ),
    retards_par_compagnie AS (
        SELECT
            origin_airport,
            airline,
            FIRST(airline_name) AS airline_name,
            COUNT(*) AS nb_vols,
            SUM(
                CASE
                    WHEN arrival_delay > 0 THEN arrival_delay
                    ELSE 0
                END
            ) AS total_retard_positif_min,
            ROUND(100.0 * AVG(CAST(is_delayed_15 AS INT)), 2) AS taux_retard_15_pct,
            PERCENTILE_APPROX(arrival_delay, 0.50) AS retard_median_min
        FROM vols
        GROUP BY origin_airport, airline
        HAVING COUNT(*) >= 300
    ),
    classement AS (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY origin_airport
                ORDER BY taux_retard_15_pct DESC, nb_vols DESC, airline
            ) AS rang,
            COUNT(*) OVER (PARTITION BY origin_airport) AS nb_compagnies_classees
        FROM retards_par_compagnie
    )
    SELECT
        a.origin_airport,
        a.origin_city,
        a.origin_state,
        a.origin_lat,
        a.origin_lon,
        a.nb_vols_aeroport,
        a.taux_retard_aeroport_pct,
        c.airline,
        c.airline_name,
        c.nb_vols,
        c.total_retard_positif_min,
        c.taux_retard_15_pct,
        c.retard_median_min,
        c.nb_compagnies_classees
    FROM aeroports a
    JOIN classement c
      ON a.origin_airport = c.origin_airport AND c.rang = 1
    ORDER BY a.nb_vols_aeroport DESC
""")

df_gold_3.coalesce(1) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(GOLD_TOP_AIRLINE_PATH)

print(f"{df_gold_3.count()} aéroports (codes IATA valides, au moins 1 000 vols)")
df_gold_3.show(15, truncate=False)

224 aéroports (codes IATA valides, au moins 1 000 vols)


+--------------+-----------------+------------+----------+----------+----------------+------------------------+-------+----------------------------+-------+------------------------+------------------+-----------------+----------------------+
|origin_airport|origin_city      |origin_state|origin_lat|origin_lon|nb_vols_aeroport|taux_retard_aeroport_pct|airline|airline_name                |nb_vols|total_retard_positif_min|taux_retard_15_pct|retard_median_min|nb_compagnies_classees|
+--------------+-----------------+------------+----------+----------+----------------+------------------------+-------+----------------------------+-------+------------------------+------------------+-----------------+----------------------+
|ATL           |Atlanta          |GA          |33.64044  |-84.42694 |343506          |16.44                   |NK     |Spirit Air Lines            |4310   |111160                  |37.75             |5                |11                    |
|ORD           |Chicago         

**Interprétation**

- Une fois les codes d'octobre exclus, il reste **224 vrais aéroports** avec au moins 1 000 vols (la première version de cette table comptait aussi environ 290 faux aéroports à codes numériques).
- **La compagnie la plus en retard d'un aéroport l'est nettement plus que la moyenne de cet aéroport** : 3,5 points d'écart en médiane. Le choix de la compagnie compte donc même quand on part du même aéroport.
- **Ce sont souvent les mêmes compagnies** : trois compagnies régionales, American Eagle (dans 58 aéroports), Skywest (46) et Atlantic Southeast (44), puis les low-cost Spirit (19) et Frontier (13). Dans les grands hubs (Atlanta, Chicago O'Hare, Dallas, Denver, Los Angeles, Houston), c'est Spirit, avec 28 à 39 % de vols en retard.
- Le classement se fait au taux de retard et non au total de minutes : au total de minutes, la plus grosse compagnie de chaque aéroport arrivait presque toujours en tête, simplement parce qu'elle a le plus de vols.

In [6]:
# Fichiers avant / après OPTIMIZE
avant = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").select("numFiles", "sizeInBytes").first()
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").select("metrics.numFilesAdded", "metrics.numFilesRemoved").show()
apres = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").select("numFiles", "sizeInBytes").first()
print(f"Fichiers de la table Silver : {avant.numFiles} avant, {apres.numFiles} après OPTIMIZE "
      f"({apres.sizeInBytes / 1024**2:.1f} Mo, blocs HDFS de 16 Mo)")

+-------------+---------------+
|numFilesAdded|numFilesRemoved|
+-------------+---------------+
|            1|              2|
+-------------+---------------+



Fichiers de la table Silver : 2 avant, 1 après OPTIMIZE (213.9 Mo, blocs HDFS de 16 Mo)


OPTIMIZE regroupe les petits fichiers de la table Delta en fichiers plus grands. Sur HDFS, chaque fichier et chaque bloc ajoute des métadonnées gérées par le NameNode. Une trop grande quantité de petits fichiers alourdit donc ces métadonnées et augmente le coût des opérations de lecture. La compaction réduit ce nombre de fichiers et facilite les lectures.

Ici, `OPTIMIZE` fusionne les 2 fichiers de la table Silver en un seul fichier de 214 Mo. Ce fichier occupe 14 blocs HDFS de 16 Mo, tous pleins sauf le dernier : le NameNode suit 1 fichier et 14 blocs, le minimum possible pour ce volume. Sur une table alimentée par de nombreux petits `MERGE`, le gain serait bien plus grand.

## 5. Tables Gold #4 et #5 : la vitrine 3D du dashboard (`flight_replay`, `route_flows`)

L'onglet « Le ciel en 3D » du dashboard rejoue en 3D les vols de quatre journées réelles et montre le réseau de l'année. Il lit deux tables Gold :

| Table | Grain | Usage |
|---|---|---|
| `flight_replay` | 1 ligne par vol des 4 journées remarquables | avions 3D animés (deck.gl) |
| `route_flows` | 1 ligne par route aéroport → aéroport sur l'année | carte 3D du réseau (arcs + colonnes) |

Difficulté technique : les horaires du dataset sont en **heure locale de chaque aéroport**. Pour que tous les avions partagent la même horloge, on convertit tout en **heure de l'Est**, avec le fuseau de l'État de départ et l'heure d'été 2015 (du 8 mars au 31 octobre). L'Arizona, Hawaï, Porto Rico, les îles Vierges, Guam et les Samoa n'appliquent pas l'heure d'été ; les États à cheval sur deux fuseaux prennent celui de leurs principaux aéroports.

In [7]:
TZ = {  # État : (décalage UTC, heure d'été)
    **{s: (-5, True) for s in ["CT", "DE", "FL", "GA", "IN", "KY", "MA", "MD", "ME", "MI", "NC", "NH", "NJ", "NY",
                               "OH", "PA", "RI", "SC", "VA", "VT", "WV", "DC"]},
    **{s: (-6, True) for s in ["AL", "AR", "IA", "IL", "KS", "LA", "MN", "MO", "MS", "ND", "NE", "OK", "SD", "TN",
                               "TX", "WI"]},
    **{s: (-7, True) for s in ["CO", "ID", "MT", "NM", "UT", "WY"]},
    "AZ": (-7, False),
    **{s: (-8, True) for s in ["CA", "NV", "OR", "WA"]},
    "AK": (-9, True), "HI": (-10, False),
    "PR": (-4, False), "VI": (-4, False), "GU": (10, False), "AS": (-11, False),
}
spark.createDataFrame([(s, h, d) for s, (h, d) in TZ.items()],
                      "state STRING, utc_offset_h INT, observes_dst BOOLEAN").createOrReplaceTempView("v_tz")
etats = {r[0] for r in spark.sql("SELECT DISTINCT origin_state FROM v_silver WHERE airport_code_valid").collect()}
assert not (etats - set(TZ)), f"États sans fuseau : {sorted(etats - set(TZ))}"

# Les 4 journées à rejouer (ROW_NUMBER)
jours = spark.sql("""
    WITH quotidien AS (
        SELECT flight_date,
               COUNT(*) AS n_vols,
               100 * AVG(CAST(is_cancelled AS INT)) AS taux_annulation,
               100 * AVG(CASE WHEN NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
                              THEN CAST(is_delayed_15 AS INT) END) AS taux_retard
        FROM v_silver
        WHERE airport_code_valid
        GROUP BY flight_date
    ),
    classes AS (
        SELECT *,
               ROW_NUMBER() OVER (ORDER BY taux_retard DESC)     AS rang_pire,
               ROW_NUMBER() OVER (ORDER BY taux_retard ASC)      AS rang_meilleur,
               ROW_NUMBER() OVER (ORDER BY taux_annulation DESC) AS rang_annulation,
               COUNT(*) OVER ()                                  AS n_jours
        FROM quotidien
    )
    SELECT flight_date, n_vols, ROUND(taux_retard, 1) AS taux_retard, ROUND(taux_annulation, 1) AS taux_annulation,
           CASE WHEN rang_pire = 1                         THEN 'La journée la plus en retard'
                WHEN rang_annulation = 1                   THEN 'Le jour du blizzard'
                WHEN rang_pire = CAST(n_jours / 2 AS INT)  THEN 'Une journée ordinaire'
                WHEN rang_meilleur = 1                     THEN 'La journée la plus ponctuelle'
           END AS day_label,
           CASE WHEN rang_pire = 1 THEN 1 WHEN rang_annulation = 1 THEN 2
                WHEN rang_pire = CAST(n_jours / 2 AS INT) THEN 3 WHEN rang_meilleur = 1 THEN 4 END AS day_order
    FROM classes
""").filter("day_label IS NOT NULL").orderBy("day_order")
jours.show(truncate=False)
assert jours.count() == 4, "Une même journée cumule deux records : ajustez les critères"
jours.createOrReplaceTempView("v_jours")

26/11/19 14:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

+-----------+------+-----------+---------------+-----------------------------+---------+
|flight_date|n_vols|taux_retard|taux_annulation|day_label                    |day_order|
+-----------+------+-----------+---------------+-----------------------------+---------+
|2015-01-04 |16352 |49.3       |2.6            |La journée la plus en retard |1        |
|2015-01-27 |15155 |7.1        |19.0           |Le jour du blizzard          |2        |
|2015-09-03 |16835 |17.7       |0.5            |Une journée ordinaire        |3        |
|2015-09-01 |16253 |6.9        |0.5            |La journée la plus ponctuelle|4        |
+-----------+------+-----------+---------------+-----------------------------+---------+



26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

In [8]:
# flight_replay : décollage = départ prévu + retard + roulage ; atterrissage = décollage + air_time
df_replay = spark.sql("""
    WITH base AS (
        SELECT s.*, j.day_label, j.day_order,
               s.flight_date BETWEEN DATE'2015-03-08' AND DATE'2015-10-31' AS heure_ete,
               o.utc_offset_h AS tz_o, o.observes_dst AS dst_o
        FROM v_silver s
        JOIN v_jours j ON s.flight_date = j.flight_date
        JOIN v_tz o    ON s.origin_state = o.state
        WHERE s.airport_code_valid
    ),
    horaires AS (
        SELECT *,
               (CAST(scheduled_departure / 100 AS INT) * 60 + scheduled_departure % 100)
                 - 60 * (tz_o + IF(heure_ete AND dst_o, 1, 0))
                 + 60 * (-5 + IF(heure_ete, 1, 0))            AS sched_dep_et
        FROM base
    )
    SELECT flight_date, day_label, day_order, airline, airline_name, flight_number,
           origin_airport, origin_city, origin_lat, origin_lon,
           destination_airport, dest_city, dest_lat, dest_lon, distance,
           sched_dep_et,
           sched_dep_et + departure_delay + taxi_out            AS takeoff_et,
           sched_dep_et + departure_delay + taxi_out + air_time AS landing_et,
           departure_delay, arrival_delay, is_delayed_15, is_cancelled, is_diverted
    FROM horaires
""")
df_replay.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save("/lakehouse/gold/flight_replay")

# route_flows : routes d'au moins 100 vols
df_routes = spark.sql("""
    SELECT origin_airport, FIRST(origin_city) AS origin_city, FIRST(origin_lat) AS origin_lat, FIRST(origin_lon) AS origin_lon,
           destination_airport, FIRST(dest_city) AS dest_city, FIRST(dest_lat) AS dest_lat, FIRST(dest_lon) AS dest_lon,
           CAST(PERCENTILE_APPROX(distance, 0.5) AS INT)        AS distance,
           COUNT(*)                                             AS n_vols,
           ROUND(100 * AVG(CAST(is_delayed_15 AS INT)), 1)      AS taux_retard,
           PERCENTILE_APPROX(arrival_delay, 0.5)                AS retard_median
    FROM v_silver
    WHERE airport_code_valid AND NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
    GROUP BY origin_airport, destination_airport
    HAVING COUNT(*) >= 100
""")
df_routes.coalesce(1).write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save("/lakehouse/gold/route_flows")

# Contrôles
chk = spark.read.format("delta").load("/lakehouse/gold/flight_replay")
chk.groupBy("day_order", "day_label").agg(
    F.count("*").alias("n_vols"),
    F.expr("percentile_approx(landing_et - takeoff_et, 0.5)").alias("duree_mediane_min"),
    F.expr("percentile_approx(takeoff_et, array(0.05, 0.5, 0.95))").alias("decollage_p5_p50_p95_min"),
).orderBy("day_order").show(truncate=False)
print("route_flows :", spark.read.format("delta").load("/lakehouse/gold/route_flows").count(), "routes")

26/11/19 14:47:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

26/11/19 14:47:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 14:47:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/19 1

+---------+-----------------------------+------+-----------------+------------------------+
|day_order|day_label                    |n_vols|duree_mediane_min|decollage_p5_p50_p95_min|
+---------+-----------------------------+------+-----------------+------------------------+
|1        |La journée la plus en retard |16352 |100              |[440, 936, 1440]        |
|2        |Le jour du blizzard          |15155 |85               |[445, 894, 1354]        |
|3        |Une journée ordinaire        |16835 |92               |[429, 899, 1398]        |
|4        |La journée la plus ponctuelle|16253 |91               |[426, 890, 1372]        |
+---------+-----------------------------+------+-----------------+------------------------+



route_flows : 4075 routes


**Lecture**

- Les deux records sont en janvier, ce qui confirme l'effet de l'hiver vu en J2 et dans la table mensuelle.
- **4 janvier 2015, la plus en retard** : un vol sur deux arrive avec 15 minutes de retard ou plus, mais seulement 2,6 % d'annulations. C'est le dimanche de retour des fêtes, sur un réseau chargé.
- **27 janvier 2015, le blizzard** qui a frappé le nord-est : 19 % des vols sont annulés, mais les vols maintenus arrivent presque tous à l'heure (7 % de retard seulement). Les compagnies annulent à l'avance plutôt que de laisser les retards s'accumuler.
- Retard et annulation sont donc deux réponses différentes à une perturbation : le dashboard affiche les deux.
- Dans l'animation du dashboard, on voit la vague de retard se former au fil de la journée. Un jour ordinaire (3 septembre), la part des avions en vol qui arriveront en retard passe de 3 % à 8 h à 32 % à 22 h (heure de l'Est). Le 4 janvier, elle passe de 19 % à 66 %.